<a href="https://www.kaggle.com/code/aleksandrmorozov123/ai-agents?scriptVersionId=246689840" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Implementation of reflection patterns on Groq models**

In [2]:
! pip install groq 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 4.5 MB/s eta 0:00:00


In [3]:
!pip install python-dotenv

In [4]:
import os
import re
import math
import json
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("Groq")

# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()

client = Groq(api_key=secret_value_0)

In [5]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

In [6]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Bucket Sort algorithm"
    }
)

In [7]:
bucketsort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": bucketsort_code
    }
)

In [8]:
display_markdown(bucketsort_code, raw=True)

Here is a Python implementation of the Bucket Sort algorithm:
```
def bucket_sort(arr):
    """
    Sorts an array using the Bucket Sort algorithm.

    Time complexity: O(n + k) where n is the length of the array and k is the number of buckets.
    Space complexity: O(n + k)

    :param arr: The array to be sorted
    :return: The sorted array
    """
    # Create empty buckets
    buckets = [[] for _ in range(len(arr))]

    # Distribute elements into buckets
    for elem in arr:
        index = int(len(arr) * elem)
        buckets[index].append(elem)

    # Sort each bucket individually
    for bucket in buckets:
        bucket.sort()

    # Concatenate the sorted buckets
    sorted_arr = []
    for bucket in buckets:
        sorted_arr.extend(bucket)

    return sorted_arr

# Example usage:
arr = [0.34, 0.12, 0.67, 0.91, 0.45, 0.23, 0.78, 0.56]
sorted_arr = bucket_sort(arr)
print(sorted_arr)
```
This implementation assumes that the input array contains elements between 0 and 1, and uses the bucket index as a simple hash function to distribute the elements into buckets. Each bucket is then sorted individually using the built-in `sort()` method, and the sorted buckets are concatenated to produce the final sorted array.

Please let me know if you have any critiques or suggestions for improvement!

In [9]:
# reflection part
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are an experienced phD computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

In [10]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": bucketsort_code
    }
)

In [11]:
# generate critique to the python code
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [12]:
display_markdown(critique, raw=True)

Here's a detailed critique of the provided Bucket Sort implementation:

**Correctness:**
The implementation seems to be correct for the specific use case of sorting an array of floating-point numbers between 0 and 1. However, it's essential to note that this implementation has limitations, which will be discussed later.

**Code Quality and Readability:**
The code is well-structured, and the variable names are descriptive. The docstring provides a clear explanation of the function's purpose, input, and output. The time and space complexities are also correctly analyzed.

**Suggestions for Improvement:**

1. **Generalization:** The current implementation is tightly coupled with the assumption that the input array contains elements between 0 and 1. To make the implementation more flexible, you could allow the user to specify the range of values or the number of buckets. This would enable the algorithm to work with different types of data.

2. **Hash Function:** The current hash function `int(len(arr) * elem)` is simple but may not provide a good distribution of elements in the buckets, especially when the input array contains a large number of elements or the elements are not uniformly distributed. You could consider using a more robust hash function, such as `int((max_arr - min_arr) * elem / num_buckets)`, where `max_arr` and `min_arr` are the maximum and minimum values in the array, and `num_buckets` is the desired number of buckets.

3. **Bucket Size:** The current implementation creates a separate bucket for each element in the array, which can lead to poor performance when the input array is large. You could consider using a fixed number of buckets (e.g., `num_buckets = 10`) and then distribute the elements into these buckets using the hash function.

4. **Sorting Buckets:** Instead of using the built-in `sort()` method for each bucket, you could consider using a more efficient sorting algorithm, such as Insertion Sort or Merge Sort, especially when the buckets are small.

5. **Error Handling:** The implementation does not handle errors or edge cases. You could add input validation to ensure that the input array is not empty and contains only numerical values.

Here's an updated version of the implementation that addresses these suggestions:
```python
def bucket_sort(arr, num_buckets=10):
    """
    Sorts an array using the Bucket Sort algorithm.

    Time complexity: O(n + k) where n is the length of the array and k is the number of buckets.
    Space complexity: O(n + k)

    :param arr: The array to be sorted
    :param num_buckets: The number of buckets (default: 10)
    :return: The sorted array
    """
    if not arr:
        return []

    min_arr = min(arr)
    max_arr = max(arr)

    # Create empty buckets
    buckets = [[] for _ in range(num_buckets)]

    # Distribute elements into buckets
    for elem in arr:
        index = int((elem - min_arr) / (max_arr - min_arr) * (num_buckets - 1))
        buckets[index].append(elem)

    # Sort each bucket individually using Insertion Sort
    for bucket in buckets:
        for i in range(1, len(bucket)):
            key = bucket[i]
            j = i - 1
            while j >= 0 and bucket[j] > key:
                bucket[j + 1] = bucket[j]
                j -= 1
            bucket[j + 1] = key

    # Concatenate the sorted buckets
    sorted_arr = []
    for bucket in buckets:
        sorted_arr.extend(bucket)

    return sorted_arr

# Example usage:
arr = [0.34, 0.12, 0.67, 0.91, 0.45, 0.23, 0.78, 0.56]
sorted_arr = bucket_sort(arr)
print(sorted_arr)
```
This updated implementation provides a more flexible and efficient Bucket Sort algorithm.

In [13]:
# add critique to the generation chat history
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

In [14]:
# generation step 2
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="llama3-70b-8192"
).choices[0].message.content

In [15]:
display_markdown(essay, raw=True)

I appreciate your detailed critique and suggestions for improvement. Your updated implementation addresses the limitations of the original code and provides a more robust and flexible Bucket Sort algorithm.

Here are some additional thoughts on the updated implementation:

**Error Handling:** While the updated implementation does not explicitly handle errors, it does include input validation to ensure that the input array is not empty. However, it's still important to consider adding explicit error handling for edge cases, such as non-numerical values in the input array.

**Bucket Size:** The updated implementation uses a fixed number of buckets (`num_buckets=10`), which can be adjusted based on the input data. However, it's worth considering dynamically adjusting the number of buckets based on the input data, such as using `num_buckets = int(len(arr) ** 0.5)`.

**Sorting Buckets:** The updated implementation uses Insertion Sort for each bucket, which is a good choice for small buckets. However, for larger buckets, it might be more efficient to use a different sorting algorithm, such as Merge Sort or Quick Sort.

**Code Readability:** The updated implementation is well-structured and easy to follow. However, some of the variable names, such as `min_arr` and `max_arr`, could be renamed to be more descriptive, such as `min_value` and `max_value`.

Here's a revised version of the implementation that addresses these additional suggestions:
```python
def bucket_sort(arr, num_buckets=10):
    """
    Sorts an array using the Bucket Sort algorithm.

    Time complexity: O(n + k) where n is the length of the array and k is the number of buckets.
    Space complexity: O(n + k)

    :param arr: The array to be sorted
    :param num_buckets: The number of buckets (default: 10)
    :return: The sorted array
    """
    if not arr:
        return []

    try:
        min_value = min(arr)
        max_value = max(arr)
    except TypeError:
        raise ValueError("Input array must contain only numerical values")

    # Create empty buckets
    buckets = [[] for _ in range(num_buckets)]

    # Distribute elements into buckets
    for elem in arr:
        index = int((elem - min_value) / (max_value - min_value) * (num_buckets - 1))
        buckets[index].append(elem)

    # Sort each bucket individually using Insertion Sort
    for bucket in buckets:
        for i in range(1, len(bucket)):
            key = bucket[i]
            j = i - 1
            while j >= 0 and bucket[j] > key:
                bucket[j + 1] = bucket[j]
                j -= 1
            bucket[j + 1] = key

    # Concatenate the sorted buckets
    sorted_arr = []
    for bucket in buckets:
        sorted_arr.extend(bucket)

    return sorted_arr

# Example usage:
arr = [0.34, 0.12, 0.67, 0.91, 0.45, 0.23, 0.78, 0.56]
sorted_arr = bucket_sort(arr)
print(sorted_arr)
```
This revised implementation includes explicit error handling for non-numerical values in the input array and uses more descriptive variable names.

**Implement tool function for agent**

In [16]:
import json

def get_current_weather(location: str, unit: str):
	"""
	Get the current weather in a given location

	location (str): The city and state, e.g. Miami, Florida
	unit (str): The unit. It can take two values; "celsius", "fahrenheit"
	"""
	if location == "Miami":
		return json.dumps({"Fahrenheit": 75, "unit": unit})

	else:
		return json.dumps({"temperature": 130, "unit": unit})

In [17]:
get_current_weather(location="Miami", unit="Fahrenheit")

'{"Fahrenheit": 75, "unit": "Fahrenheit"}'

In [18]:
MODEL = "llama-3.3-70b-versatile"
GROQ_CLIENT = Groq(api_key=secret_value_0)

# Define the System Prompt as a constant
TOOL_SYSTEM_PROMPT = """
You are a function calling AI model. You are provided with function signatures within <tools></tools> XML tags. 
You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug 
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.
For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>}
</tool_call>

Here are the available tools:

<tools> {
    "name": "get_current_weather",
    "description": "Get the current weather in a given location location (str): The city and state, e.g. Madrid, Barcelona unit (str): The unit. It can take two values; 'celsius', 'fahrenheit'",
    "parameters": {
        "properties": {
            "location": {
                "type": "str"
            },
            "unit": {
                "type": "str"
            }
        }
    }
}
</tools>
"""

In [19]:
# ask the question
tool_chat_history = [
    {
        "role": "system",
        "content": TOOL_SYSTEM_PROMPT
    }
]
agent_chat_history = []

user_msg = {
    "role": "user",
    "content": "What's the current temperature in Miami, in Fahrenheit?"
}

tool_chat_history.append(user_msg)
agent_chat_history.append(user_msg)

output = GROQ_CLIENT.chat.completions.create(
    messages=tool_chat_history,
    model=MODEL
).choices[0].message.content

print(output)

<tool_call>
{"name": "get_current_weather","arguments": {"location": "Miami", "unit": "fahrenheit"}}
</tool_call>


In [20]:
def parse_tool_call_str(tool_call_str: str):
    pattern = r'</?tool_call>'
    clean_tags = re.sub(pattern, '', tool_call_str)
    
    try:
        tool_call_json = json.loads(clean_tags)
        return tool_call_json
    except json.JSONDecodeError:
        return clean_tags
    except Exception as e:
        print(f"Unexpected error: {e}")
        return "There was some error parsing the Tool's output"

In [21]:
parsed_output = parse_tool_call_str(output)
parsed_output

{'name': 'get_current_weather',
 'arguments': {'location': 'Miami', 'unit': 'fahrenheit'}}

In [22]:
result = get_current_weather(**parsed_output["arguments"])

In [23]:
result

'{"Fahrenheit": 75, "unit": "fahrenheit"}'

In [24]:
agent_chat_history.append({
    "role": "user",
    "content": f"Observation: {result}"
})

In [25]:
GROQ_CLIENT.chat.completions.create(
    messages=agent_chat_history,
    model=MODEL
).choices[0].message.content

'The current temperature in Miami is 75°F.'

**Implementation of planning pattern**

In [26]:
!pip install agents

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.6/118.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 24.2 MB/s eta 0:00:00
  Created wheel for agents: filename=agents-1.4.0-py3-none-any.whl size=62714 sha256=81a4ee059ca988f9a7945f4b7f34c192bd6e50ba119bdea4acbb9f2f28d26ddd
  Stored in directory: /root/.cache/pip/wheels/6a/41/e1/4d212ede7c9752d8f7050d78fa3295304bc22b8fb530d9ebe4
Successfully built agents


In [27]:
!pip install agentic_patterns

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 0.28.0
    Uninstalling groq-0.28.0:
      Successfully uninstalled groq-0.28.0


In [28]:
from agentic_patterns.tool_pattern.tool import tool
from agentic_patterns.utils.extraction import extract_tag_content
MODEL = "llama-3.3-70b-versatile"

In [29]:
# Define the System Prompt as a constant
REACT_SYSTEM_PROMPT = """
You are a function calling AI model. You operate by running a loop with the following steps: Thought, Action, Observation.
You are provided with function signatures within <tools></tools> XML tags.
You may call one or more functions to assist with the user query. Don' make assumptions about what values to plug
into functions. Pay special attention to the properties 'types'. You should use those types as in a Python dict.

For each function call return a json object with function name and arguments within <tool_call></tool_call> XML tags as follows:

<tool_call>
{"name": <function-name>,"arguments": <args-dict>, "id": <monotonically-increasing-id>}
</tool_call>

Here are the available tools / actions:

<tools> 
%s
</tools>

Example session:

<question>What's the current temperature in Miami?</question>
<thought>I need to get the current weather in Miami</thought>
<tool_call>{"name": "get_current_weather","arguments": {"location": "Miami", "unit": "fahrenheit"}, "id": 0}</tool_call>

You will be called again with this:

<observation>{0: {"temperature": 75, "unit": "fahrenheit"}}</observation>

You then output:

<response>The current temperature in Miami is 75 degrees Fahrenheit</response>

Additional constraints:

- If the user asks you something unrelated to any of the tools above, answer freely enclosing your answer with <response></response> tags.
"""